In [0]:
%pip install torch-geometric-signed-directed s3fs

In [0]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import f1_score
from torch_geometric.loader import DataLoader

# ============================================================
# Parameters (notebook widgets)
# ============================================================
dbutils.widgets.text("tag", "v1", "Experiment tag")
dbutils.widgets.text("n_users", "20", "Number of users")

EXPERIMENT_TAG = dbutils.widgets.get("tag")
N_USERS = int(dbutils.widgets.get("n_users"))

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}"
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"

# Storage mode: True = S3 user cache, False = local
USE_S3_STORAGE = True

S3_GNN_BUCKET = "pablocelayes-test"
S3_USER_CACHE_PREFIX = "learning/sna-classifier-gnn/user_samples_cache"

print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")
print(f"User cache: s3://{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}")

In [0]:
HISTORY_PATH = f"{EXPERIMENT_DIR}/training_history.pkl"

if not os.path.exists(HISTORY_PATH):
    raise FileNotFoundError(f"No history found at {HISTORY_PATH}")

with open(HISTORY_PATH, "rb") as f:
    history = pickle.load(f)

print(f"Loaded history: {len(history['step'])} checkpoints, {len(history['epoch_step'])} epochs")
print(f"  Last checkpoint step: {history['step'][-1] if history['step'] else 'N/A'}")
print(f"  Best val F1 (checkpoint): {max(history['val_f1']):.4f}" if history['val_f1'] else "")
print(f"  Best val F1 (end-of-epoch): {max(history['epoch_val_f1']):.4f}" if history['epoch_val_f1'] else "")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Loss curves ---
ax = axes[0]
if history["step"]:
    ax.plot(history["step"], history["train_loss"], 'b-', alpha=0.6, label='Train loss (checkpoint)')
    ax.plot(history["step"], history["val_loss"], 'r-', alpha=0.6, label='Val loss (checkpoint)')
if history["epoch_step"]:
    ax.plot(history["epoch_step"], history["epoch_val_loss"], 'ro-', markersize=5, label='Val loss (end-of-epoch)')
ax.set_xlabel('Global Step')
ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Right: F1 curves ---
ax = axes[1]
if history["step"]:
    ax.plot(history["step"], history["val_f1"], 'r-', alpha=0.6, label='Val F1 (checkpoint)')
if history["epoch_step"]:
    ax.plot(history["epoch_step"], history["epoch_train_f1"], 'b^-', markersize=5, label='Train F1 (end-of-epoch)')
    ax.plot(history["epoch_step"], history["epoch_val_f1"], 'ro-', markersize=5, label='Val F1 (end-of-epoch)')
ax.set_xlabel('Global Step')
ax.set_ylabel('F1 Score')
ax.set_title('Training & Validation F1')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle(f'GNN Training Curves — {FINAL_TAG}', fontsize=13)
plt.tight_layout()
plt.show()

In [0]:
from gnn_models import PretrainedEmbeddingLookup, RetweetDataset, ParquetGNNLoader, RetweetGNN, evaluate

DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
# Load best model
BEST_MODEL_PATH = f"{EXPERIMENT_DIR}/best_retweet_gnn_general.pt"
assert os.path.exists(BEST_MODEL_PATH), f"No model found at {BEST_MODEL_PATH}"

model = RetweetGNN(
    ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH, device=device
).to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True, map_location=device))
model.eval()
print(f"Loaded model from {BEST_MODEL_PATH}")

# ---------------------------------------------------------------------------
# Load the SAME validation set used during training (cached by 3.1)
# ---------------------------------------------------------------------------
import gc
import json
from torch_geometric.data import Batch

VAL_CACHE_PATH = os.path.join(EXPERIMENT_DIR, "val_samples_cache")
assert os.path.isdir(VAL_CACHE_PATH), f"No val cache found at {VAL_CACHE_PATH}"

# Read metadata
with open(os.path.join(VAL_CACHE_PATH, "metadata.json")) as f:
    val_meta = json.load(f)

# Simple streaming loader over cached chunks (same logic as CachedValLoader in 3.1)
class _CachedChunkLoader:
    """Loads precomputed PyG Data chunks from disk and yields Batch objects."""
    def __init__(self, cache_path, metadata, batch_size=256):
        self.batch_size = batch_size
        self._total_samples = metadata["total_samples"]
        self._chunk_files = sorted(
            os.path.join(cache_path, f)
            for f in os.listdir(cache_path)
            if f.startswith("chunk_") and f.endswith(".pt")
        )

    @property
    def total_samples(self):
        return self._total_samples

    def __len__(self):
        return self._total_samples // self.batch_size

    def __iter__(self):
        carry_over = []
        for chunk_path in self._chunk_files:
            chunk_data = torch.load(chunk_path, map_location="cpu", weights_only=False)
            samples = carry_over + chunk_data
            del chunk_data
            start = 0
            while start + self.batch_size <= len(samples):
                yield Batch.from_data_list(samples[start : start + self.batch_size])
                start += self.batch_size
            carry_over = samples[start:]
            del samples
            gc.collect()

BATCH_SIZE = 256
val_loader = _CachedChunkLoader(VAL_CACHE_PATH, val_meta, batch_size=BATCH_SIZE)
print(f"Val loader (from training cache): {val_loader.total_samples} samples, "
      f"{len(val_loader)} batches, {len(val_loader._chunk_files)} chunks")
print(f"  Label counts: {val_meta.get('label_counts', 'N/A')}")

In [0]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Evaluate on the same validation set used during training
global_f1, global_preds, global_labels, _ = evaluate(model, val_loader, device)
print(f"=== GNN Validation F1: {global_f1:.4f} ===")
print(f"  Total val samples: {val_loader.total_samples}")
print(f"  Positive rate: {global_labels.float().mean():.3f}")

In [0]:
# Compute F1 per user on their val samples (from S3 user cache, same splits as 3.1)
import s3fs
import pyarrow.parquet as pq

_loader_fs = s3fs.S3FileSystem() if USE_S3_STORAGE else None

# Load the user_sample.json from the experiment (same users as 3.1 training)
SAMPLE_PATH = f"{EXPERIMENT_DIR}/user_sample.json"
assert os.path.exists(SAMPLE_PATH), f"No user sample found at {SAMPLE_PATH}"
with open(SAMPLE_PATH) as f:
    user_sample = json.load(f)

TRAIN_GROUPS = ["u_train", "au_train"]
TEST_GROUPS = [g for g in user_sample if g not in TRAIN_GROUPS]

# Val split logic from 3.1: train-group users' "test" split + test-group users' both splits
val_user_splits_list = []  # (uid, split_name)
for group in TRAIN_GROUPS:
    for uid in user_sample.get(group, []):
        val_user_splits_list.append((uid, "test"))
for group in TEST_GROUPS:
    for uid in user_sample.get(group, []):
        val_user_splits_list.append((uid, "train"))
        val_user_splits_list.append((uid, "test"))

print(f"Per-user val evaluation: {len(set(uid for uid, _ in val_user_splits_list))} users, "
      f"{len(val_user_splits_list)} user/split combos")

gnn_f1s = {}

for uid_str, split_name in val_user_splits_list:
    path = f"{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}/{uid_str}/{split_name}.snappy.parquet"
    if not _loader_fs.exists(path):
        continue
    with _loader_fs.open(path, 'rb') as f:
        table = pq.read_table(f)
    if table.num_rows == 0:
        continue

    samples = []
    for row_idx in range(table.num_rows):
        edge_src = table.column('edge_src')[row_idx].as_py()
        edge_dst = table.column('edge_dst')[row_idx].as_py()
        if edge_src:
            edge_index = np.column_stack([
                np.array(edge_src, dtype=np.int32),
                np.array(edge_dst, dtype=np.int32),
            ])
        else:
            edge_index = np.empty((0, 2), dtype=np.int32)
        samples.append({
            "central_user_id": int(table.column('central_user_id')[row_idx].as_py()),
            "neighbor_ids": np.array(table.column('neighbor_ids')[row_idx].as_py(), dtype=np.int64),
            "retweeted_ids": np.array(table.column('retweeted_ids')[row_idx].as_py(), dtype=np.int64),
            "edge_index": edge_index,
            "label": int(table.column('label')[row_idx].as_py()),
        })
    del table

    user_ds = RetweetDataset(samples)
    user_loader = DataLoader(user_ds, batch_size=32, shuffle=False)
    user_f1, _, _, _ = evaluate(model, user_loader, device)
    # Accumulate per-user (may have multiple splits per user — keep best? or last?)
    uid_int = int(uid_str)
    if uid_int not in gnn_f1s:
        gnn_f1s[uid_int] = []
    gnn_f1s[uid_int].append((split_name, user_f1, len(samples)))

# Aggregate: weighted F1 across splits for users with multiple entries
gnn_f1_per_user = {}
for uid, entries in gnn_f1s.items():
    total_samples = sum(n for _, _, n in entries)
    weighted_f1 = sum(f1 * n for _, f1, n in entries) / total_samples
    gnn_f1_per_user[uid] = weighted_f1

gnn_f1s = gnn_f1_per_user  # keep same variable name for downstream cells
gnn_f1_values = list(gnn_f1s.values())

print(f"\n=== GNN — Per-user Val F1 Distribution ===")
print(f"  Mean:   {np.mean(gnn_f1_values):.4f}")
print(f"  Median: {np.median(gnn_f1_values):.4f}")
print(f"  Std:    {np.std(gnn_f1_values):.4f}")
print(f"  Min:    {np.min(gnn_f1_values):.4f}")
print(f"  Max:    {np.max(gnn_f1_values):.4f}")
print(f"  Users:  {len(gnn_f1_values)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(gnn_f1_values, bins=20, edgecolor='black', alpha=0.7, color='darkorange')
ax.axvline(np.mean(gnn_f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(gnn_f1_values):.3f}')
ax.axvline(np.median(gnn_f1_values), color='blue', linestyle='--', label=f'Median: {np.median(gnn_f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title(f'GNN (General) — Per-user Val F1 Distribution — {FINAL_TAG}')
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# Load baseline SVC results from shared per-user cache (independent of GNN experiment)
BASELINE_USER_CACHE_PATH = f"{DATA_PATH}/baseline_svc_user_cache.pkl"

if not os.path.exists(BASELINE_USER_CACHE_PATH):
    print(f"No baseline user cache found at {BASELINE_USER_CACHE_PATH} — skipping comparison.")
else:
    with open(BASELINE_USER_CACHE_PATH, "rb") as f:
        baseline_user_cache = pickle.load(f)

    # Assemble experiment-level results from cache for users that overlap with GNN
    baseline_users = [uid for uid in gnn_f1s.keys() if uid in baseline_user_cache]
    baseline_f1s = {uid: baseline_user_cache[uid]["f1"] for uid in baseline_users}
    all_baseline_test_preds = [
        (baseline_user_cache[uid]["preds"], baseline_user_cache[uid]["labels"])
        for uid in baseline_users
    ]
    print(f"Loaded baseline cache: {len(baseline_user_cache)} users total, "
          f"{len(baseline_users)} overlap with GNN experiment")

    # Combined baseline F1
    all_bl_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
    all_bl_labels = np.concatenate([l for _, l in all_baseline_test_preds])
    combined_f1 = f1_score(all_bl_labels, all_bl_preds)

    # Side-by-side comparison
    common_users = set(baseline_f1s.keys()) & set(gnn_f1s.keys())
    baseline_common = [baseline_f1s[u] for u in common_users]
    gnn_common = [gnn_f1s[u] for u in common_users]

    print(f"=== Comparison (on {len(common_users)} common users) ===")
    print(f"  Baseline SVC mean F1: {np.mean(baseline_common):.4f}")
    print(f"  GNN mean F1:          {np.mean(gnn_common):.4f}")
    print(f"  GNN wins: {sum(g > b for g, b in zip(gnn_common, baseline_common))}/{len(common_users)}")
    print(f"\n  Baseline combined F1 (pooled): {combined_f1:.4f}")
    print(f"  GNN val F1 (pooled):           {global_f1:.4f}")

    # Side-by-side histogram
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

    axes[0].hist(baseline_common, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(np.mean(baseline_common), color='red', linestyle='--', label=f'Mean: {np.mean(baseline_common):.3f}')
    axes[0].set_xlabel('Test F1 Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Baseline SVC (RBF)')
    axes[0].legend()

    axes[1].hist(gnn_common, bins=20, edgecolor='black', alpha=0.7, color='darkorange')
    axes[1].axvline(np.mean(gnn_common), color='red', linestyle='--', label=f'Mean: {np.mean(gnn_common):.3f}')
    axes[1].set_xlabel('Test F1 Score')
    axes[1].set_title('GNN (General)')
    axes[1].legend()

    plt.suptitle(f'Per-user Val F1 Distribution: Baseline vs GNN — {FINAL_TAG}', fontsize=13)
    plt.tight_layout()
    plt.show()